# Analyze Encounter Leave-Out Hyperalignment

For each held-out encounter, the alignment was fit on all OTHER encounters.  
Each encounter's data is therefore independently aligned — no data leakage.

This notebook runs:

1. **ISC per encounter** — pairwise inter-subject correlation as an alignment quality check
2. **RSA** — for each encounter, compute an 8×8 RDM from the key task contrasts;  
   then second-order Spearman r between RDMs, MDS, dendrograms, and per-subject RDMs

In [ ]:
import sys
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from itertools import combinations
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.manifold import MDS
from nilearn import plotting
import nibabel as nib

try:
    _nb_dir = Path(__file__).resolve().parent
except NameError:
    _nb_dir = Path.cwd()
sys.path.insert(0, str(_nb_dir))
from config import (
    TASKS, CONTRASTS, SUBJECTS, ENCOUNTERS, RT_CONTRAST,
    RESULTS_DIR, RSA_CONTRASTS,
    setup_masker_and_labels, BASE_DIR, SESSIONS,
    build_first_level_contrast_map_path,
)

LEAVEOUT_DIR = RESULTS_DIR / 'encounter_leaveout'
FIG_DIR      = _nb_dir / 'figures' / 'encounter_leaveout'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Canonical label for each RSA contrast (task acronym)
RSA_LABELS = list(RSA_CONTRASTS.keys())   # one per task, same order as TASKS
print('RSA tasks:', RSA_LABELS)
print('Setup done.')

In [ ]:
# Set up masker for volumetric brain maps
reference_path = None
for task in TASKS:
    for contrast in CONTRASTS[task]:
        if contrast == RT_CONTRAST:
            continue
        for session in SESSIONS:
            p = build_first_level_contrast_map_path(BASE_DIR, SUBJECTS[0], session, task, contrast)
            if Path(p).exists():
                reference_path = p
                break
        if reference_path: break
    if reference_path: break

masker, labels = setup_masker_and_labels(nib.load(reference_path))
print(f'Masker ready.  Reference: {reference_path}')

In [ ]:
# Load all encounter leave-out results
# tc_order is a list of (task, contrast) tuples — NO encounter dimension
# (each enc{N}/ directory stores the held-out encounter's data, already sliced by encounter)
data = {}   # enc_key -> {'masked': {subj: arr}, 'aligned': {subj: arr}, 'tc_order': list}
for enc in ENCOUNTERS:
    enc_dir = LEAVEOUT_DIR / f'enc{enc}'
    if not enc_dir.exists():
        print(f'  Missing: enc{enc}')
        continue
    with open(enc_dir / 'tc_order.pkl', 'rb') as f:
        tc_order = pickle.load(f)
    data[enc] = {
        'tc_order': tc_order,
        'masked':  {s: np.load(enc_dir / f'masked_{s}.npy')  for s in SUBJECTS},
        'aligned': {s: np.load(enc_dir / f'aligned_{s}.npy') for s in SUBJECTS},
    }
    arr_shape = data[enc]['aligned'][SUBJECTS[0]].shape
    print(f'  enc{enc}: {len(tc_order)} (task, contrast) pairs | shape {arr_shape}')

## 1. ISC per Encounter

Pairwise Pearson r per voxel, averaged across all subject pairs,  
for both unaligned and aligned encounter data.

In [ ]:
def voxelwise_pearson(A, B):
    A = A - A.mean(axis=0, keepdims=True)
    B = B - B.mean(axis=0, keepdims=True)
    denom = np.sqrt((A**2).sum(0) * (B**2).sum(0)) + 1e-10
    return (A * B).sum(0) / denom

pairs = list(combinations(SUBJECTS, 2))
isc_before_per_enc, isc_after_per_enc = {}, {}

for enc, d in data.items():
    before_vals, after_vals = [], []
    for s1, s2 in pairs:
        before_vals.append(voxelwise_pearson(d['masked'][s1],  d['masked'][s2]))
        after_vals.append( voxelwise_pearson(d['aligned'][s1], d['aligned'][s2]))
    isc_before_per_enc[enc] = np.mean(before_vals, axis=0)
    isc_after_per_enc[enc]  = np.mean(after_vals,  axis=0)
    print(f'enc{enc}  before={isc_before_per_enc[enc].mean():.3f}  '
          f'after={isc_after_per_enc[enc].mean():.3f}  '
          f'gain={(isc_after_per_enc[enc] - isc_before_per_enc[enc]).mean():.3f}')

In [ ]:
# Line plot: mean ISC across encounters, before and after alignment
enc_list = sorted(data.keys())
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(enc_list, [isc_before_per_enc[e].mean() for e in enc_list],
        'o-', color='steelblue', label='Before')
ax.plot(enc_list, [isc_after_per_enc[e].mean()  for e in enc_list],
        'o-', color='coral',     label='After')
ax.set_xlabel('Encounter'); ax.set_ylabel('Mean ISC (Pearson r)')
ax.set_xticks(enc_list)
ax.set_title('ISC per encounter (leave-one-out cross-validated)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'isc_per_encounter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Brain map: ISC gain (after - before) for each encounter
for enc in enc_list:
    diff_map = isc_after_per_enc[enc] - isc_before_per_enc[enc]
    vmax = np.percentile(np.abs(diff_map), 98)
    diff_img = masker.inverse_transform(diff_map)
    display = plotting.plot_stat_map(
        diff_img, display_mode='z', cut_coords=6,
        cmap='RdBu_r', vmax=vmax, colorbar=True,
        title=f'ISC gain: encounter {enc}',
    )
    display.savefig(FIG_DIR / f'isc_gain_enc{enc}.png')
    plt.close('all')

## 2. RSA: RDMs per Encounter

For each encounter, extract the rows corresponding to the 8 key contrasts  
(one per task from `RSA_CONTRASTS`), average across subjects, and compute  
a pairwise correlation-distance RDM (8 × 8).

In [ ]:
def compute_rdm(matrix):
    """Pairwise 1 - Pearson r among rows of matrix (n_items, n_voxels)."""
    return squareform(pdist(matrix, metric='correlation'))

def upper_tri(M):
    idx = np.triu_indices(M.shape[0], k=1)
    return M[idx]

# For each encounter, find the row indices for the 8 RSA contrasts
# tc_order is a list of (task, contrast) tuples
rdms_aligned  = {}   # enc -> (n_tasks, n_tasks)
rdms_masked   = {}   # enc -> (n_tasks, n_tasks)
missing_tasks = {}

for enc, d in data.items():
    tc_order = d['tc_order']  # list of (task, contrast) tuples
    tc_to_row = {tc: i for i, tc in enumerate(tc_order)}

    rsa_rows = []
    rsa_found_labels = []
    missing = []
    for task in RSA_LABELS:
        contrast = RSA_CONTRASTS[task]
        key = (task, contrast)
        if key in tc_to_row:
            rsa_rows.append(tc_to_row[key])
            rsa_found_labels.append(task)
        else:
            missing.append(task)
    if missing:
        print(f'  enc{enc}: missing tasks {missing}')
    missing_tasks[enc] = missing

    # Average aligned/masked maps across subjects for these rows
    aligned_mean = np.mean(
        [d['aligned'][s][rsa_rows] for s in SUBJECTS], axis=0
    )   # (n_found, n_voxels)
    masked_mean  = np.mean(
        [d['masked'][s][rsa_rows]  for s in SUBJECTS], axis=0
    )

    rdms_aligned[enc] = (compute_rdm(aligned_mean), rsa_found_labels)
    rdms_masked[enc]  = (compute_rdm(masked_mean),  rsa_found_labels)

    print(f'enc{enc}: RDM shape {rdms_aligned[enc][0].shape}')

In [ ]:
# Plot RDM heatmaps for each encounter (aligned)
fig, axes = plt.subplots(1, len(enc_list), figsize=(4 * len(enc_list), 4))
if len(enc_list) == 1:
    axes = [axes]

for ax, enc in zip(axes, enc_list):
    rdm, lbl = rdms_aligned[enc]
    im = ax.imshow(rdm, cmap='viridis', vmin=0, vmax=2)
    ax.set_xticks(range(len(lbl))); ax.set_yticks(range(len(lbl)))
    ax.set_xticklabels(lbl, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(lbl, fontsize=7)
    ax.set_title(f'Enc {enc}')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('RDMs per encounter (aligned, group average)', fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / 'rdm_heatmaps_per_encounter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dendrograms per encounter
fig, axes = plt.subplots(1, len(enc_list), figsize=(4 * len(enc_list), 4))
if len(enc_list) == 1:
    axes = [axes]

for ax, enc in zip(axes, enc_list):
    rdm, lbl = rdms_aligned[enc]
    Z = linkage(upper_tri(rdm), method='average')
    dendrogram(Z, labels=lbl, ax=ax, leaf_rotation=45, leaf_font_size=8)
    ax.set_title(f'Enc {enc}')

fig.suptitle('Hierarchical clustering per encounter (aligned)', fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / 'dendrograms_per_encounter.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Second-Order RSA: How Similar are RDMs Across Encounters?

Compare each encounter's RDM to every other encounter's RDM  
using the Spearman r of their upper-triangular vectors.

In [ ]:
n_enc = len(enc_list)
second_order = np.zeros((n_enc, n_enc))

for i, enc_i in enumerate(enc_list):
    for j, enc_j in enumerate(enc_list):
        rdm_i, lbl_i = rdms_aligned[enc_i]
        rdm_j, lbl_j = rdms_aligned[enc_j]
        # Use only tasks present in both encounters
        common = [t for t in lbl_i if t in lbl_j]
        if len(common) < 3:
            second_order[i, j] = np.nan
            continue
        idx_i = [lbl_i.index(t) for t in common]
        idx_j = [lbl_j.index(t) for t in common]
        rdm_i_sub = rdm_i[np.ix_(idx_i, idx_i)]
        rdm_j_sub = rdm_j[np.ix_(idx_j, idx_j)]
        r, _ = spearmanr(upper_tri(rdm_i_sub), upper_tri(rdm_j_sub))
        second_order[i, j] = r

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(second_order, cmap='hot', vmin=0, vmax=1)
ax.set_xticks(range(n_enc)); ax.set_yticks(range(n_enc))
ax.set_xticklabels([f'Enc{e}' for e in enc_list])
ax.set_yticklabels([f'Enc{e}' for e in enc_list])
plt.colorbar(im, ax=ax, label='Spearman r')
ax.set_title('Second-order RSA: RDM similarity across encounters')
plt.tight_layout()
plt.savefig(FIG_DIR / 'second_order_rsa.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# MDS on encounter RDMs
dist_matrix = 1 - second_order
np.fill_diagonal(dist_matrix, 0)
dist_matrix = np.clip(dist_matrix, 0, None)  # numerical safety

mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, normalized_stress='auto')
coords = mds.fit_transform(dist_matrix)

fig, ax = plt.subplots(figsize=(5, 5))
for i, enc in enumerate(enc_list):
    ax.scatter(coords[i, 0], coords[i, 1], s=100)
    ax.annotate(f'Enc {enc}', (coords[i, 0], coords[i, 1]),
                textcoords='offset points', xytext=(6, 6))
ax.set_title('MDS of encounter RDMs (2D)')
ax.set_xlabel('MDS 1'); ax.set_ylabel('MDS 2')
plt.tight_layout()
plt.savefig(FIG_DIR / 'mds_encounters.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Per-Subject RDMs

Compute each subject's RDM separately, then compute second-order Spearman r  
between subject pairs within each encounter.

In [ ]:
# Per-subject RDMs per encounter
subj_rdms = {}   # enc -> subj -> (n_tasks, n_tasks)

for enc, d in data.items():
    tc_order = d['tc_order']
    tc_to_row = {tc: i for i, tc in enumerate(tc_order)}

    rsa_rows, rsa_found_labels = [], []
    for task in RSA_LABELS:
        key = (task, RSA_CONTRASTS[task])
        if key in tc_to_row:
            rsa_rows.append(tc_to_row[key])
            rsa_found_labels.append(task)

    subj_rdms[enc] = {}
    for subj in SUBJECTS:
        maps = d['aligned'][subj][rsa_rows]   # (n_found, n_voxels)
        subj_rdms[enc][subj] = (compute_rdm(maps), rsa_found_labels)

print('Per-subject RDMs computed.')

In [ ]:
# Heatmap grid: per-subject RDMs for one representative encounter
rep_enc = enc_list[0]
n_subj  = len(SUBJECTS)
ncols   = min(4, n_subj)
nrows   = int(np.ceil(n_subj / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
axes_flat = np.array(axes).flat if nrows > 1 else [axes] if ncols == 1 else axes
for ax, subj in zip(np.array(axes).flat, SUBJECTS):
    rdm, lbl = subj_rdms[rep_enc][subj]
    im = ax.imshow(rdm, cmap='viridis', vmin=0, vmax=2)
    ax.set_xticks(range(len(lbl))); ax.set_yticks(range(len(lbl)))
    ax.set_xticklabels(lbl, rotation=45, ha='right', fontsize=6)
    ax.set_yticklabels(lbl, fontsize=6)
    ax.set_title(subj, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
for ax in list(np.array(axes).flat)[n_subj:]:
    ax.axis('off')

fig.suptitle(f'Per-subject RDMs — encounter {rep_enc} (aligned)', fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / f'per_subject_rdms_enc{rep_enc}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Inter-subject RDM similarity per encounter
# For each encounter: pairwise Spearman r between all subject-pair upper triangles
n_subj = len(SUBJECTS)
subj_pairs = list(combinations(range(n_subj), 2))
mean_inter_subj_r = {}

for enc in enc_list:
    rs = []
    lbl = subj_rdms[enc][SUBJECTS[0]][1]
    for i, j in subj_pairs:
        rdm_i = subj_rdms[enc][SUBJECTS[i]][0]
        rdm_j = subj_rdms[enc][SUBJECTS[j]][0]
        r, _ = spearmanr(upper_tri(rdm_i), upper_tri(rdm_j))
        rs.append(r)
    mean_inter_subj_r[enc] = np.nanmean(rs)
    print(f'enc{enc}: mean inter-subject RDM r = {mean_inter_subj_r[enc]:.3f}')

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(enc_list, [mean_inter_subj_r[e] for e in enc_list], 'o-', color='darkorchid')
ax.set_xlabel('Encounter'); ax.set_ylabel('Mean Spearman r')
ax.set_xticks(enc_list)
ax.set_title('Inter-subject RDM similarity per encounter (aligned)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'inter_subject_rdm_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Contrast-Pair Distance Trajectories Across Encounters

For each pair of key contrasts, plot how their representational distance  
(1 − r) changes from encounter 1 to encounter 5.

In [ ]:
# Only plot pairs where both contrasts are present in every encounter
common_tasks_all_encs = set(RSA_LABELS)
for enc in enc_list:
    _, lbl = rdms_aligned[enc]
    common_tasks_all_encs &= set(lbl)
common_tasks = sorted(common_tasks_all_encs, key=RSA_LABELS.index)
n_common = len(common_tasks)
print(f'Tasks present in all encounters: {common_tasks}')

# Build trajectory: n_pairs x n_enc matrix
task_pairs = list(combinations(range(n_common), 2))
trajectories = np.zeros((len(task_pairs), len(enc_list)))

for col_idx, enc in enumerate(enc_list):
    rdm, lbl = rdms_aligned[enc]
    idx_map = {t: lbl.index(t) for t in common_tasks}
    for row_idx, (i, j) in enumerate(task_pairs):
        ti, tj = common_tasks[i], common_tasks[j]
        trajectories[row_idx, col_idx] = rdm[idx_map[ti], idx_map[tj]]

fig, ax = plt.subplots(figsize=(8, 5))
for row_idx, (i, j) in enumerate(task_pairs):
    label = f'{common_tasks[i]} vs {common_tasks[j]}'
    ax.plot(enc_list, trajectories[row_idx], 'o-', alpha=0.7, label=label)
ax.set_xlabel('Encounter')
ax.set_ylabel('Distance (1 − Pearson r)')
ax.set_xticks(enc_list)
ax.set_title('Task-pair representational distance across encounters')
ax.legend(fontsize=7, loc='upper right', ncol=2)
plt.tight_layout()
plt.savefig(FIG_DIR / 'pair_distance_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()